STEP 1

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# read counts data
expr = pd.read_csv("SRP_counts.tsv", sep="\t", index_col="Gene")
meta = pd.read_csv("metadata.tsv", sep="\t")

assert list(expr.columns) == meta.refinebio_accession_code.tolist()
print("genes x samples:", expr.shape)

genes x samples: (43363, 335)


In [4]:
meta[["region", "dx"]] = meta.refinebio_subject.str.split("_", n=1, expand=True)
meta["donor"] = meta.refinebio_title.str.split("_").str[0]

print(pd.crosstab(meta.dx, meta.region))
print("unique donors:", meta.donor.nunique())

region            ancg  dlpfc  nacc
dx                                 
bipolar disorder    25     28    30
control             28     28    27
major depression    27     30    32
schizophrenia       27     28    25
unique donors: 96


In [5]:
run_to_lib = meta.set_index("refinebio_accession_code").refinebio_title

print("duplicate runs:", run_to_lib.duplicated().sum())

expr_lib = expr.T.groupby(run_to_lib).mean().T

meta_lib = (
    meta.drop_duplicates("refinebio_title")
    .set_index("refinebio_title")
    .loc[expr_lib.columns]
)

print("genes x libraries:", expr_lib.shape)

duplicate runs: 69
genes x libraries: (43363, 266)


In [6]:
print("range before log:", expr_lib.values.min(), expr_lib.values.max())

log_expr = np.log2(expr_lib + 1)

range before log: -0.1583617006285569 197.4761356701856


STEP 2

In [7]:
# selecting variable genes
gene_variance = log_expr.var(axis=1)
top_5000 = gene_variance.nlargest(5000).index
expr_5000 = log_expr.loc[top_5000].T
print(expr_5000.shape)

(266, 5000)


In [8]:
!pip install scikit-learn-extra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 15.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-learn-extra: filename=scikit_learn_extra-0.3.0-cp313-cp313-linux_x86_64.whl size=2154562 sha256=5e56d8712ff3b869b89f07bba9258108133b5113aa6b4a20a1e0a947bab7db36
  Stored in directory: /root/.cache/pip/wheels/17/4b/b8/6b6711681d0981b110c9cc91ad6d1ebd88adf1547e1da301fc
Successfully built scikit-learn-extra


In [9]:
from sklearn_extra.cluster import KMedoids

In [13]:
# PAM with k=2
pam_2 = KMedoids(
    n_clusters=2,
    random_state=42,
    metric="euclidean"
)

clusters_2 = pam_2.fit_predict(expr_5000)
print("Cluster counts:")
print(pd.Series(clusters_2).value_counts().sort_index())

Cluster counts:
0     89
1    177
Name: count, dtype: int64


In [14]:
k_values = [2, 3, 4, 5, 6]

pam_results = {}

for k in k_values:
    pam = KMedoids(
        n_clusters=k,
        random_state=42,
        metric="euclidean"
    )

    pam_results[k] = pam.fit_predict(expr_5000)

    print(f"k = {k}")
    print(pd.Series(pam_results[k]).value_counts().sort_index())
    print()

k = 2
0     89
1    177
Name: count, dtype: int64

k = 3
0    86
1    89
2    91
Name: count, dtype: int64

k = 4
0    87
1    67
2    53
3    59
Name: count, dtype: int64

k = 5
0    87
1    45
2    30
3    54
4    50
Name: count, dtype: int64

k = 6
0    34
1    50
2    26
3    36
4    44
5    76
Name: count, dtype: int64



In [12]:
gene_numbers = [10, 100, 1000, 5000, 10000]

gene_sets = {}

for n in gene_numbers:
    top_genes = gene_variance.nlargest(n).index
    gene_sets[n] = log_expr.loc[top_genes].T

    print(n, gene_sets[n].shape)

10 (266, 10)
100 (266, 100)
1000 (266, 1000)
5000 (266, 5000)
10000 (266, 10000)


In [15]:
comparison_2_3 = pd.crosstab(
    pam_results[2],
    pam_results[3]
)

print(comparison_2_3)

col_0   0   1   2
row_0            
0      86   3   0
1       0  86  91


In [16]:
def compare_clusters(k1, k2):
    return pd.crosstab(
        pam_results[k1],
        pam_results[k2]
    )

print("k = 2 vs k = 3")
print(compare_clusters(2, 3))

print("\nk = 3 vs k = 4")
print(compare_clusters(3, 4))

print("\nk = 4 vs k = 5")
print(compare_clusters(4, 5))

print("\nk = 5 vs k = 6")
print(compare_clusters(5, 6))

k = 2 vs k = 3
col_0   0   1   2
row_0            
0      86   3   0
1       0  86  91

k = 3 vs k = 4
col_0   0   1   2   3
row_0                
0      86   0   0   0
1       1  58   1  29
2       0   9  52  30

k = 4 vs k = 5
col_0   0   1   2   3   4
row_0                    
0      87   0   0   0   0
1       0  35   1  30   1
2       0   4  28   4  17
3       0   6   1  20  32

k = 5 vs k = 6
col_0   0   1   2   3   4   5
row_0                        
0      11   0   0   0   0  76
1       5  40   0   0   0   0
2       0   0  25   0   5   0
3      18   0   0  35   1   0
4       0  10   1   1  38   0


In [17]:
gene_counts = [10, 100, 1000, 10000]

pam_gene_results = {}

for n in gene_counts:
    pam_gene_results[n] = {}

    for k in k_values:
        pam = KMedoids(
            n_clusters=k,
            random_state=42,
            metric="euclidean"
        )

        pam_gene_results[n][k] = pam.fit_predict(gene_sets[n])

        print(f"{n} genes, k = {k}")
        print(pd.Series(pam_gene_results[n][k]).value_counts().sort_index())
        print()

10 genes, k = 2
0     85
1    181
Name: count, dtype: int64

10 genes, k = 3
0    87
1    94
2    85
Name: count, dtype: int64

10 genes, k = 4
0    85
1    94
2    77
3    10
Name: count, dtype: int64

10 genes, k = 5
0    84
1    79
2    77
3    10
4    16
Name: count, dtype: int64

10 genes, k = 6
0     9
1    74
2    61
3    24
4    14
5    84
Name: count, dtype: int64

100 genes, k = 2
0     86
1    180
Name: count, dtype: int64

100 genes, k = 3
0     61
1    180
2     25
Name: count, dtype: int64

100 genes, k = 4
0    35
1    89
2    51
3    91
Name: count, dtype: int64

100 genes, k = 5
0    22
1    64
2    64
3    77
4    39
Name: count, dtype: int64

100 genes, k = 6
0    54
1    74
2    22
3    69
4    36
5    11
Name: count, dtype: int64

1000 genes, k = 2
0     87
1    179
Name: count, dtype: int64

1000 genes, k = 3
0    86
1    94
2    86
Name: count, dtype: int64

1000 genes, k = 4
0    85
1    33
2    80
3    68
Name: count, dtype: int64

1000 genes, k = 5
0    33
1  

In [18]:
from scipy.stats import chi2_contingency

gene_comparisons = [
    (10, 100),
    (10, 1000),
    (10, 10000),
    (100, 1000),
    (100, 10000),
    (1000, 10000)
]

chi_results = []

for k in k_values:
    for g1, g2 in gene_comparisons:

        contingency_table = pd.crosstab(
            pam_gene_results[g1][k],
            pam_gene_results[g2][k]
        )

        chi2, p, dof, expected = chi2_contingency(contingency_table)

        chi_results.append({
            "k": k,
            "Genes 1": g1,
            "Genes 2": g2,
            "Chi-square": chi2,
            "p-value": p,
            "Degrees of freedom": dof
        })

chi_results_df = pd.DataFrame(chi_results)

print(chi_results_df)

    k  Genes 1  Genes 2  Chi-square        p-value  Degrees of freedom
0   2       10      100  256.928661   8.016961e-58                   1
1   2       10     1000  252.539738   7.257130e-57                   1
2   2       10    10000  256.928661   8.016961e-58                   1
3   2      100     1000  256.982992   7.801288e-58                   1
4   2      100    10000  261.448835   8.292932e-59                   1
5   2     1000    10000  256.982992   7.801288e-58                   1
6   3       10      100  261.511221   2.154413e-55                   4
7   3       10     1000  286.849545   7.431122e-61                   4
8   3       10    10000  273.523361   5.549901e-58                   4
9   3      100     1000  266.000000   2.322409e-56                   4
10  3      100    10000  266.000000   2.322409e-56                   4
11  3     1000    10000  418.945613   2.240330e-89                   4
12  4       10      100  312.404680   6.084506e-62                   9
13  4 

In [24]:
# Save PAM k=2 cluster assignments for the 5,000-gene analysis

pam_labels_5000 = pd.Series(
    pam_results[2],
    index=expr_5000.index,
    name="g5000_k2"
)

pam_labels_5000.to_frame().to_csv(
    "a3_labels_pam.tsv",
    sep="\t"
)

print("Saved:", "a3_labels_pam.tsv")
print(pam_labels_5000.value_counts().sort_index())

Saved: a3_labels_pam.tsv
g5000_k2
0     89
1    177
Name: count, dtype: int64
